# 01_dblink 데모 (v2) — 외부 DB 매칭

Kim 2022 개체들을 외부 DB에 매칭. **API 키 불필요.** 인터넷만 있으면 `Run All`.

- ingredient → PubChem → (없으면) **Wikidata 폴백 + 교차 ID 수확**
- method → CHMO (OLS)
- equipment → OBI/CHMO (OLS) → (없으면) Wikidata 폴백
- vendor → **Wikidata (회사 타입 필터)**

> 같은 폴더의 `dblink.py` 를 불러온다. 01_dblink 폴더에서 실행.

In [ ]:
from dblink import match_entity, match_vendor
print("불러오기 성공")

In [ ]:
samples = [
    ("benzylphosphonic acid", "ingredient"),
    ("caesium bromide", "ingredient"),
    ("methylammonium bromide", "ingredient"),   # PubChem MISS -> Wikidata 폴백 기대
    ("X-ray photoelectron spectroscopy", "method"),
    ("transmission electron microscopy", "method"),
    ("scanning electron microscope", "equipment"),
    ("mass spectrometer", "equipment"),
]
for surface, category in samples:
    rec = match_entity(surface, category)
    if rec["normalized_ids"]:
        ids = ", ".join(f"{x['db']} {x['id']}" for x in rec["normalized_ids"])
    else:
        ids = "(미매칭 -> LLM 단계로)"
    print(f"[{rec['provenance'] or '-':11s}] {surface:34s} {category:10s} {ids}")

## vendor(제조사) — Wikidata 회사 타입 필터
동음이의(예: 'Kratos'=아이돌 앨범, 'Gatan'=성씨)를 설명 키워드로 걸러낸다.
다만 같은 이름의 회사가 여럿이면 정확한 자회사까지는 못 집을 수 있음(한계).

In [ ]:
for v in ["Kratos", "Hamamatsu Photonics", "Gatan", "Sigma-Aldrich", "없는회사123"]:
    r = match_vendor(v)
    print(f"{v:22s} -> {r['qid'] or '(미매칭)'}  {r['label'] or ''}")

## 메모
- **미매칭 = 실패 아님.** established가 아닐 뿐, 이후 LLM 단계로.
- PubChem이 놓친 *methylammonium bromide* 를 Wikidata 폴백이 잡고 **PubChem CID·CAS까지 역수확**하는 걸 확인하라.
- vendor는 '같은 이름 회사 존재 확인' 수준(정확한 자회사 식별은 어려움) → seed 사전 + Wikidata 검증 병행 권장.
- 결과는 `_dblink_cache.json` 에 캐시됨.